# Sabatini & Kaufman (2024) on DANDI:000070: dynamics (notebook 1 of 2)

Paper: D. A. Sabatini and M. T. Kaufman, "Reach-dependent reorientation of rotational dynamics in
motor cortex", Nat. Commun. 15:7007 (2024), doi:10.1038/s41467-024-51308-7. The claims file
(`claims/sabatini-2024.yaml`) was written from the version of record (PMC11325044) and its
supplement.

Dataset: DANDI:000070, "Neural population dynamics during reaching", version **0.261122.2046**
(doi:10.48324/dandi.000070/0.261122.2046). The paper cites the dandiset without a version (it was
published in August 2024, before the first published version of December 2025), so the most recent
published version is pinned. The two sessions the paper's Data availability statement names are read:
`sub-Jenkins_ses-20090918` and `sub-Nitschke_ses-20100923`. The file names of this version match the
paper's (the Nitschke date is 2010 in both).

This notebook tests the claims on the temporal basis factorization (TBF), single-condition dynamics,
jPCA and the geometry of the rotations. Notebook 2 (`reproduce_sabatini_2024_kinematics.ipynb`) tests
the claims on prediction, encoding and decoding.

| claim | notebook |
|---|---|
| data-composition, fig3g-basis-frequencies, tbf-variance-explained, tbf-shuffle-control, fig3h-recoverability, fig3i-duration-warping, jpca-fit-r2, jpca-variance-explained, fig4d-jpca-variance-correlation, fig4f-plane-alignment-explains-jpca, single-lds-variance, fig2c-condition-lds-variance, fig3a-eigenvalue-clusters, fig3c-eigenvalue-distance-auc, suppfig2-eigenvalues-from-kinematics, suppfig3-model-comparison, fig4b-plane-alignment, suppfig6a-location-alignment, fig5a-frequencies-segregated, suppfig6c-location-distinct-from-rotations, fig4c-subspace-excursion-angles, suppfig6b-location-dimensionality, rotation-orientation-dimensionality | tested here |
| fig5c, fig6b, fig6c, fig6e, fig7c (both), suppfig7a, fig7f-ldr-decoding, trial-curvature-decoding, suppfig8 | tested in notebook 2 |
| fig7f-decoder-beats-standard | tested in notebook 2 (GPFA decoders not run) |

In [ ]:
import time, resource, warnings
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import ldr
warnings.filterwarnings("ignore", category=RuntimeWarning)
np.set_printoptions(precision=3, suppress=True, linewidth=150)
T_START = time.time()
SEED_OFFSETS = [0, 1000, 2000, 3000, 4000]
DS = ldr.build("all")
NAMES = ldr.DATASETS
MONKEYS = sorted({n.split("-")[1] for n in NAMES})
def pmap(fn, args):
    t = time.time()
    out = [fn(*a) for a in args]
    print(f"[{fn.__name__}: {len(args)} jobs in {time.time() - t:.0f} s]")
    return out
print("loaded in %.0f s, peak memory %.1f GB" % (time.time() - T_START,
      resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6))

In [ ]:
# Data check: what was loaded, and the description against the values.
for m in MONKEYS:
    d = DS[f"M1-{m}"]
    tab = d["table"]
    keep = (tab.maze_condition > 0) & (tab.maze_condition % 3 != 0)
    print(f"monkey {m}: {d['n_trials_file']} trials in the file; maze conditions "
          f"{tab.maze_condition.min()}..{tab.maze_condition.max()} ({tab.maze_condition.nunique()} values, "
          f"{(tab.maze_condition == 0).sum()} trials with condition 0)")
    for r in range(3):
        sub = tab[(tab.maze_condition > 0) & (tab.maze_condition % 3 == r)]
        print(f"   condition mod 3 = {r}: {sub.maze_condition.nunique()} conditions, targets "
              f"{sorted(sub.maze_num_targets.unique())}, barriers {sorted(sub.maze_num_barriers.unique())}, "
              f"trial_version {sorted(sub.trial_version.unique())}")
    succ = keep & (tab.task_success == 1)
    print(f"   successful trials of the 72 conditions: {succ.sum()}; with delay > 400 ms: "
          f"{(succ & (tab.delay > 0.4)).sum()}; discard_trial set: {(tab.discard_trial == 1).sum()}; "
          f"windows past the trial end dropped: {d['dropped']}")
    n_per = np.bincount(d["cond"], minlength=72)
    print(f"   trials per condition {n_per.min()}..{n_per.max()} (median {np.median(n_per):.0f})")
    print(f"   delay range {tab.delay.min()*1000:.0f}..{tab.delay.max()*1000:.0f} ms; offline minus online "
          f"movement onset, median {np.median(tab.offline_move_onset_time - tab.move_begins_time)*1000:.0f} ms")
    hk = d["kin"]
    print(f"   hand displacement at +850 ms (mm, start-subtracted): |x| up to {np.abs(hk[:, -1, 0]).max():.0f}, "
          f"|y| up to {np.abs(hk[:, -1, 1]).max():.0f}; peak speed of the condition means "
          f"{np.linalg.norm(np.diff(hk, axis=1), axis=2).max() * 100:.0f} mm/s")
for n in NAMES:
    d = DS[n]
    rate = d["X"].mean((0, 2))
    print(f"{n}: {len(d['units'])} units (threshold-crossing channels), mean rate {rate.min():.1f}.."
          f"{rate.max():.1f} spikes/s, trials {d['Xt'].shape[0]}, conditions {d['X'].shape[0]}")

In [ ]:
# Clock check. Hand speed in the condition means peaks after the trial table's movement onset, and the
# reach direction can be decoded from spikes after onset and not with shuffled labels. The
# appears; the window before onset is in the delay period, when the target is already shown, so the
# shuffled labels are the null.
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import cross_val_score
for n in NAMES:
    d = DS[n]
    sp = np.linalg.norm(np.diff(d["kin"], axis=1), axis=2) * 100
    tpk = ldr.TIMES[1:][sp.argmax(1)]
    ang = np.arctan2(d["kin"][:, -1, 1], d["kin"][:, -1, 0])
    octant = (np.floor((ang + np.pi) / (2 * np.pi) * 8) % 8).astype(int)[d["cond"]]
    c = d["counts"]
    post = c[:, :, 355:655].sum(2)          # 0 to +300 ms
    pre = c[:, :, 0:300].sum(2)             # -355 to -55 ms (delay period: target shown, preparatory activity)
    g = ldr.rng("clock-check")
    shuffled = g.permutation(octant)
    acc = [cross_val_score(LinearDiscriminantAnalysis(), np.sqrt(x), y, cv=5).mean()
           for x, y in ((post, octant), (pre, octant), (post, shuffled))]
    print(f"{n}: hand speed peak {np.median(tpk):.0f} ms after onset (range {tpk.min()}..{tpk.max()}); "
          f"8-way direction decoding: 0..300 ms {acc[0]:.2f}, -355..-55 ms {acc[1]:.2f}, shuffled labels "
          f"{acc[2]:.2f} (chance 0.125)")

In [ ]:
# Recordings deposited more than once: trial counts, first trial times and spike counts of every
# session of the version (metadata and one unit only).
rows = []
r = ldr.requests.get(f"{ldr.API}/dandisets/{ldr.DANDISET}/versions/{ldr.VERSION}/assets/",
                     params={"page_size": 100}, timeout=60).json()["results"]
for a in sorted(r, key=lambda a: a["path"]):
    h = ldr.h5py.File(ldr.remfile.File(f"{ldr.API}/assets/{a['asset_id']}/download/"), "r")
    t = h["intervals/trials"]
    idx = h["units/spike_times_index"][:]
    rows.append(dict(path=a["path"].split("/")[1][:28], trials=len(t["start_time"]),
                     first_go=round(float(t["go_cue_time"][0]), 4), units=len(idx),
                     spikes_unit0=int(idx[0]), spikes_total=int(idx[-1]),
                     conds=len(np.unique(t["maze_condition"][:])) if "maze_condition" in t else None))
    h.close()
sess = pd.DataFrame(rows)
print(sess.to_string(index=False))
dup = sess.duplicated(["trials", "spikes_unit0", "spikes_total"], keep=False)
print("sessions sharing trial and spike counts:", int(dup.sum()))

In [ ]:
# The units are unsorted threshold crossings, one per electrode, where the paper analysed sorted
# single units and stable multi-units (79 to 118 per array).
h = ldr.open_nwb(ldr.SESSIONS["J"])
print(h["units"].attrs["description"])
print("units:", len(h["units/id"]), "| columns:", list(h["units"].attrs["colnames"]))
loc = h["general/extracellular_ephys/electrodes/location"][:].astype(str)
print("electrodes per location:", dict(zip(*np.unique(loc, return_counts=True))))
h.close()

## data-composition
> Two monkeys (J and N) performed the maze task with 72 conditions (36 straight, 36 curved); a PMd and
> an M1 array in each monkey gave 79 to 118 units per array; 1469 trials for monkey J and 2014 for monkey N.

In [ ]:
expected = {"J": 1469, "N": 2014}
for m in MONKEYS:
    tab = DS[f"M1-{m}"]["table"]
    for reading in ("all", "delay400"):
        sel = ldr.select_trials(tab, reading)
        mc = tab.maze_condition[sel]
        ns, nc = (mc % 3 == 1).groupby(mc).first().sum(), (mc % 3 == 2).groupby(mc).first().sum()
        n = int(sel.sum())
        print(f"monkey {m}, reading {reading}: {mc.nunique()} conditions ({ns} straight = no barrier, "
              f"{nc} curved = barriers), {n} trials; paper {expected[m]} (within 10%: "
              f"{abs(n - expected[m]) <= 0.1 * expected[m]})")
for n in NAMES:
    print(f"{n}: {len(DS[n]['units'])} units, none marked single- or multi-unit (all threshold crossings)")

**Result: partially reproduced.** Under both trial readings each monkey has 72 conditions, 36 without barriers (straight) and 36 with barriers (curved). Monkey J has 1595 successful trials (1381 with delay above 400 ms) and N has 2121 (1884), all within 10% of 1469 and 2014. Each array has 96 units, but they are unsorted threshold crossings, one per electrode; the dataset marks no single or multi-units, so the paper's 79 to 118 sorted units per array cannot be counted.

## fig3g-basis-frequencies
> The cross-validated temporal basis functions contained 3 to 4 rotations at 0.5, 1.5, 2.5 and 4 Hz,
> along with an offset.

Number of basis functions by cross-validation (code crossValidateDimensionalityBasisFxns.m: halves of the
trials, 100 folds, k = 1 to 20, an even best k raised by one), then de-mixing (eigTransform.m).

In [ ]:
def tbf_cv_job(n, k):
    # 100 folds under the claim's seed; 20 under the four others, for time
    return ldr.cv_tbf_k(DS[n]["Xt"], DS[n]["cond"], ldr.rng("fig3g-basis-frequencies", k), folds=100 if k == 0 else 20)
jobs = [(n, k) for n in NAMES for k in SEED_OFFSETS]
CVALL = dict(zip(jobs, pmap(tbf_cv_job, jobs)))
CV = {}
for n in NAMES:
    res = [CVALL[(n, k)] for k in SEED_OFFSETS]
    CV[n] = res[0]
    print(f"{n}: held-out VE by k (claim seed) {np.round(res[0]['mean'][:12], 3)}")
    print(f"     best k {res[0]['best']} -> k = {res[0]['k']}; k under the five seeds {[r['k'] for r in res]}")
K = {n: CV[n]["k"] for n in NAMES}
TBF = {n: ldr.eig_transform(DS[n]["X"], K[n]) for n in NAMES}
targets = np.array([0.5, 1.5, 2.5, 4.0])
for n in NAMES:
    tb = TBF[n]
    gr = ldr.groups(tb)
    rot = [f for f, i in gr if len(i) == 2]
    real = [f for f, i in gr if len(i) == 1]
    near = [targets[np.argmin(abs(targets - f))] for f in rot]
    ok = [abs(f - t) <= 0.1 * t for f, t in zip(rot, near)]
    print(f"{n}: k={K[n]}; {len(rot)} rotation pairs at {np.round(rot, 2)} Hz (half-lives "
          f"{np.round([tb['halflife'][i[0]] for f, i in gr if len(i) == 2], 2)} s), {len(real)} real mode(s) at "
          f"{np.round(real, 2)} Hz; nearest of 0.5/1.5/2.5/4: {near}; within 10%: {ok}")
fig, ax = plt.subplots(1, 4, figsize=(14, 2.6))
for a, n in zip(ax, NAMES):
    a.plot(ldr.TIMES, TBF[n]["B"]); a.set_title(n); a.set_xlabel("ms from movement onset")
plt.tight_layout(); plt.show()
print("basis functions plotted are within -1..1:", all(np.nanmax(abs(TBF[n]['B'])) <= 1 for n in NAMES))

**Result: partially reproduced.** Cross-validation chooses k = 7 for both J arrays and k = 9 for both N arrays (the same under all five seeds), giving 3 (J) or 4 (N) rotation pairs and one real mode at 0 Hz in every dataset, as the claim states. The frequencies are not those of the paper: the slowest pair is at 0.61 to 0.73 Hz in all four datasets, and only 6 of the 14 rotations are within 10% of one of 0.5, 1.5, 2.5 and 4 Hz.

## tbf-variance-explained
> The shared rotations of the temporal basis factorization explained 90 to 96% of the population variance.

In [ ]:
VE_TBF = {}
for n in NAMES:
    X = DS[n]["X"]; tb = TBF[n]
    va = ldr.var_explained(X, ldr.reconstruct(tb["L"], tb["B"]))            # (a) -150..850, de-mixed
    vb = ldr.var_explained(X, ldr.reconstruct(tb["Lraw"], tb["Braw"]))      # (b) whole window, raw SVD
    VE_TBF[n] = va
    print(f"{n}: (a) mean {va.mean()*100:.1f}% (s.d. {va.std(ddof=1)*100:.1f}); (b) mean {vb.mean()*100:.1f}% "
          f"(s.d. {vb.std(ddof=1)*100:.1f}); in 81..100%: {0.81 <= va.mean() <= 1}, {0.81 <= vb.mean() <= 1}")

**Result: reproduced.** The factorization explains 92.3%, 95.1%, 97.1% and 96.1% (M1-J, PMd-J, M1-N, PMd-N) from -150 to +850 ms, and 92.5% to 97.7% over the whole window with the raw basis, all inside 81 to 100%.

## tbf-shuffle-control and fig3h-recoverability
> Shuffling time bins within each trial before smoothing lowered the variance explained by the
> factorization to 35 to 46%, significantly below the real data.

> Projections of each condition's activity recovered the temporal basis functions almost perfectly
> (91 to 95% variance explained, chance 63 to 80%).

Per trial the 1 ms bins are permuted in time, the same permutation for every unit (code scrambleTrials.m),
then smoothed and averaged; TBF with the real data's k on the raw SVD basis over the whole window, as
the code's quantifyTBFStats.m does (it calls getBasisFxns, not eigTransform). Recoverability by Eq. 6.

In [ ]:
def recoverability(X, L, B):
    ok = ~np.isnan(B[:, 0])
    Bt = B[ok].T
    out = np.zeros(len(X))
    for c in range(len(X)):
        Bh = np.linalg.pinv(L[c]) @ X[c][:, ok]
        out[c] = 1 - (Bh - Bt).var(1, ddof=1).sum() / Bt.var(1, ddof=1).sum()
    return np.clip(out, 0, None)

def scrambled_average(d, g):
    c = d["counts"][:, :, 5:1206]
    out = np.zeros((c.shape[0], c.shape[1], ldr.NT), np.float32)
    for i in range(c.shape[0]):
        p = g.permutation(c.shape[2])
        x = np.zeros((c.shape[1], 1210), np.uint8)
        x[:, 5:1206] = c[i][:, p]
        out[i] = ldr.smooth(x)
    return ldr.cond_average(out, d["cond"])

def shuffle_job(n, k):
    Xs = scrambled_average(DS[n], ldr.rng("tbf-shuffle-control", k))
    Bs, Ls = ldr.get_basis(Xs, K[n])
    return ldr.var_explained(Xs, Ls @ Bs.T), recoverability(Xs, Ls, Bs)

jobs = [(n, k) for n in NAMES for k in SEED_OFFSETS]
SHOUT = dict(zip(jobs, pmap(shuffle_job, jobs)))
SHUF = {}
for n in NAMES:
    X = DS[n]["X"]
    Br, Lr = ldr.get_basis(X, K[n])
    rows = [SHOUT[(n, k)] for k in SEED_OFFSETS]
    SHUF[n] = dict(real_ve=ldr.var_explained(X, Lr @ Br.T), real_rec=recoverability(X, Lr, Br),
                   ve=rows[0][0], rec=rows[0][1],
                   ve_seeds=[r[0].mean() for r in rows], rec_seeds=[r[1].mean() for r in rows])

In [ ]:
for n in NAMES:
    s = SHUF[n]
    p_rs = stats.ranksums(s["real_ve"], s["ve"], alternative="two-sided").pvalue
    p_sr = stats.wilcoxon(s["real_ve"], s["ve"]).pvalue
    print(f"{n}: real {s['real_ve'].mean()*100:.1f}%, shuffled {s['ve'].mean()*100:.1f}% "
          f"(s.d. {s['ve'].std(ddof=1)*100:.1f}; five seeds {np.round(np.array(s['ve_seeds'])*100, 1)}); "
          f"in 31.5..50.6%: {0.315 <= s['ve'].mean() <= 0.506}; rank-sum p = {p_rs:.2g}, signed-rank p = {p_sr:.2g}")

**Result: reproduced.** With 1 ms bins shuffled within each trial the factorization explains 35.1%, 35.4%, 45.6% and 45.5%, all inside 31.5 to 50.6% and under all five seeds, against 92.5% to 97.7% on the real data (rank-sum p = 3.9e-25 in each dataset).

In [ ]:
for n in NAMES:
    s = SHUF[n]
    tb = TBF[n]
    rec_dm = recoverability(DS[n]["X"], tb["L"], tb["B"])
    p_rs = stats.ranksums(s["real_rec"], s["rec"]).pvalue
    print(f"{n}: recoverability {s['real_rec'].mean()*100:.1f}% (s.d. {s['real_rec'].std(ddof=1)*100:.1f}; "
          f"de-mixed basis -150..850: {rec_dm.mean()*100:.1f}%), chance {s['rec'].mean()*100:.1f}% (five seeds "
          f"{np.round(np.array(s['rec_seeds'])*100, 1)}); real in 81.9..100: {s['real_rec'].mean() >= 0.819}; "
          f"chance in 56.7..88: {0.567 <= s['rec'].mean() <= 0.88}; rank-sum p = {p_rs:.2g}, real higher: "
          f"{s['real_rec'].mean() > s['rec'].mean()}")

**Result: reproduced.** Recoverability is 95.1% to 96.6% (inside 81.9 to 100%) against a shuffled chance of 84.8% to 87.8% (inside 56.7 to 88%, near its top and above the paper's unwidened 63 to 80%), real higher with rank-sum p below 1e-24 in every dataset.

## fig3i-duration-warping
> Before warping, rotation fits were unrelated or weakly related to reach duration (M1-N rho = -0.26,
> p = 0.023; others -0.18 to 0.21, p > 0.067); warping reaches to equal duration induced a negative
> correlation (rho -0.63 to -0.52, p < 0.001).

Code warpTrials.m and warpTrial.m. Reach duration is the time from movement onset to the last 10 ms bin
where hand speed is at least a tenth of its peak. The criterion says per trial; the code computes it for each
trial from its condition's mean kinematics (warpTrials.m sets `kinematics(trial) = kinematicsCond(condNum)`).
Both readings are run: (a) each trial's own hand speed, the criterion's wording, with a condition's duration
the mean over its trials (the code's `oldDuration`); (b) the condition-mean hand speed, as the code does.
Each spike gets a uniform random offset within its 1 ms bin. warpTrial.m adds a lag of 100 ms (`lag = 100`)
and scales the span from 0 to the duration in lagged time, which is from 100 ms before onset to 100 ms before
the end of the reach, to 600 ms; later activity is shifted (reading "truncate": cut at +850 ms, as Figure 3i;
variant "250": the post-reach activity scaled to 250 ms). Variant "end+100", which the code does not run:
the scaled span ends at the end of the reach itself. TBF on the raw SVD basis with the original k;
recoverability per condition.


In [ ]:
def last_fast(kin):
    """Code warpTrials.m: 10*(fastInds(end)-36), the last 10 ms bin of the speed (diff of position) at
    least a tenth of its peak, in ms from onset. kin: (..., 121, 2)."""
    sp = np.linalg.norm(np.diff(kin, axis=-2), axis=-1)
    fast = sp >= sp.max(-1, keepdims=True) / 10
    last = sp.shape[-1] - 1 - np.argmax(fast[..., ::-1], axis=-1)
    return 10 * (last + 1 - 36)

def warped_average(d, dur, g, mode="truncate", end=0):
    """Code warpTrial.m; dur: one duration per trial (ms). end=100 is the end+100 variant."""
    c = d["counts"][:, :, 5:1206]
    out = np.zeros((c.shape[0], c.shape[1], ldr.NT), np.float32)
    for i in range(c.shape[0]):
        wf = dur[i] + end
        uu, tt = np.nonzero(c[i])
        reps = c[i][uu, tt]
        uu = np.repeat(uu, reps); tt = np.repeat(tt, reps)
        st = tt + g.random(len(tt)) - 350 + 100      # code: spikeTime - 350 + lag
        w = st.copy()
        a = (st > 0) & (st < wf)
        w[a] = 600 * st[a] / wf
        b = st >= wf
        if mode == "truncate":
            w[b] = st[b] - wf + 600
        else:  # post-reach activity scaled to 250 ms, then as recorded
            post = 850 + 100 - wf
            w[b] = 600 + (st[b] - wf) * 250 / max(post, 1)
        w = w + 350 - 100
        keep = (w >= 0) & (w < 1201)
        x = np.zeros((c.shape[1], 1210), np.uint16)
        np.add.at(x, (uu[keep], 5 + np.floor(w[keep]).astype(int)), 1)
        out[i] = ldr.smooth(x)
    return ldr.cond_average(out, d["cond"])

def durations(d, reading):
    """Per-trial durations and per-condition mean durations under reading 'trial' (a) or 'cond' (b)."""
    if reading == "trial":
        dt = last_fast(d["hand10"])
    else:
        dt = last_fast(d["kin"])[d["cond"]]
    return dt, np.array([dt[d["cond"] == c].mean() for c in range(72)])

def warp_job(n, rd, k, mode, end=0):
    d = DS[n]
    dt, dc = durations(d, rd)
    Xw = warped_average(d, dt, ldr.rng("fig3i-duration-warping", k), mode, end)
    Bw, Lw = ldr.get_basis(Xw, K[n])
    return stats.pearsonr(abs(dc - dc.mean()), recoverability(Xw, Lw, Bw))

jobs = ([(n, rd, k, "truncate") for n in NAMES for rd in ("trial", "cond") for k in SEED_OFFSETS] +
        [(n, rd, 0, "250") for n in NAMES for rd in ("trial", "cond")] +
        [(n, rd, 0, "truncate", 100) for n in NAMES for rd in ("trial", "cond")])
WOUT = dict(zip(jobs, pmap(warp_job, jobs)))
WARP = {}
for n in NAMES:
    for rd in ("trial", "cond"):
        dt, dc = durations(DS[n], rd)
        r0 = stats.pearsonr(abs(dc - dc.mean()), SHUF[n]["real_rec"])
        WARP[(n, rd)] = dict(dur=dc, dur_trial=dt, r0=r0,
                             res={"truncate": [WOUT[(n, rd, k, "truncate")] for k in SEED_OFFSETS],
                                  "250": [WOUT[(n, rd, 0, "250")]], "end+100": [WOUT[(n, rd, 0, "truncate", 100)]]})


In [ ]:
for rd, lab in (("trial", "reading (a), each trial's own duration"), ("cond", "reading (b), the condition-mean duration (code)")):
    print(lab)
    for n in NAMES:
        w = WARP[(n, rd)]
        rt = w["res"]["truncate"][0]
        print(f"  {n}: durations per trial {w['dur_trial'].min():.0f}..{w['dur_trial'].max():.0f} ms, per condition "
              f"{w['dur'].min():.0f}..{w['dur'].max():.0f} (mean {w['dur'].mean():.0f}); unwarped rho = {w['r0'][0]:.3f} "
              f"(p = {w['r0'][1]:.3g}); warped (truncate) rho = {rt[0]:.3f} (p = {rt[1]:.3g}), five seeds "
              f"{np.round([v[0] for v in w['res']['truncate']], 3)}; 250 ms rho = {w['res']['250'][0][0]:.3f} "
              f"(p = {w['res']['250'][0][1]:.3g}); end+100 rho = {w['res']['end+100'][0][0]:.3f} "
              f"(p = {w['res']['end+100'][0][1]:.3g})")
    r0n = WARP[("M1-N", rd)]["r0"]
    unw = [-0.198 <= WARP[(n, rd)]["r0"][0] <= 0.231 for n in NAMES if n != "M1-N"]
    for mode in ("truncate", "250", "end+100"):
        wok = [(-0.693 <= WARP[(n, rd)]["res"][mode][0][0] <= -0.468) and WARP[(n, rd)]["res"][mode][0][1] < 0.001 for n in NAMES]
        print(f"  {mode}: warped in -0.693..-0.468 with p < 0.001: {wok}")
    print(f"  M1-N unwarped within -0.286..-0.234 and p < 0.05: {-0.286 <= r0n[0] <= -0.234 and r0n[1] < 0.05}; "
          f"other unwarped in -0.198..0.231: {unw}")
dur_a, dur_b = WARP[("M1-J", "trial")]["dur"], WARP[("M1-J", "cond")]["dur"]
print(f"monkey J: correlation of the two readings' condition durations {np.corrcoef(dur_a, dur_b)[0, 1]:.3f}; "
      f"monkey N: {np.corrcoef(WARP[('M1-N', 'trial')]['dur'], WARP[('M1-N', 'cond')]['dur'])[0, 1]:.3f}")


**Result: not reproduced.** The M1-N unwarped correlation is far from the paper's -0.26 under both readings of the duration: -0.008 (p = 0.95) with each trial's own duration (reading a) and 0.013 (p = 0.92) with the condition-mean duration (reading b, the code). The other three unwarped correlations are inside -0.198 to 0.231 under reading (a) (-0.116, -0.124, 0.210) and two of three under (b) (PMd-N 0.246). Warped with truncation, as Figure 3i, no correlation lies in -0.693 to -0.468 under reading (a) (-0.156, -0.018, -0.446, -0.304) and two of four under (b) (-0.508 and -0.504 for M1, -0.060 and -0.267 for PMd), so the partial condition fails under both. In the 250 ms variant all four lie in the range under reading (a) (-0.645, -0.559, -0.481, -0.571) and two under (b); in the end+100 variant two (a) and three (b) do. The warped correlations move by at most 0.004 over the five seeds.


## jPCA claims
jPCA as in getRotations.m: activity reconstructed from the de-mixed basis, mean-subtracted across
conditions, soft-normalised (range + 5), jPCA on bins 20 to 40 with the codepack's defaults (6 PCs, its own
soft normalisation and mean subtraction). The jPCA codepack is not in the paper's repository; jPCA is
implemented here from its published description (Churchland et al. 2012), in `ldr.jpca`.

### jpca-fit-r2
> Rotational dynamics found by jPCA fit activity within the jPCA planes with R2 of 0.64 to 0.73, higher on
> straight reaches (0.73 to 0.88) than on curved reaches (0.56 to 0.70).

In [ ]:
PLANES = {n: (2 if n.startswith("M1") else 1) for n in NAMES}
ROT = {}
straight, curved = np.arange(0, 72, 2), np.arange(1, 72, 2)
rows = []
for n in NAMES:
    tb = TBF[n]
    ROT[n] = ldr.get_rotations(tb["L"], tb["B"], PLANES[n])
    rs = ldr.get_rotations(tb["L"], tb["B"], PLANES[n], conds=straight)
    rc = ldr.get_rotations(tb["L"], tb["B"], PLANES[n], conds=curved)
    # per-condition R2 of the projected trajectory in the top plane (signed-rank test, reported)
    def per_cond(r):
        jp = r["jp"]; out = []
        Xn = (r["Xr"] - r["mean"]) * r["sn"][None, :, None]
        for c in range(len(Xn)):
            z = jp["jPCs_highD"][:, :2].T @ Xn[c][:, ldr.T0:40]
            dz = np.diff(z, axis=1); M2 = jp["J"][:, :2].T @ jp["Mskew"] @ jp["J"][:, :2]
            fit = (z[:, :-1].T @ M2).T
            out.append(1 - ((dz - fit) ** 2).sum() / ((dz - dz.mean(1, keepdims=True)) ** 2).sum())
        return np.array(out)
    ps, pc = per_cond(rs), per_cond(rc)
    p = stats.wilcoxon(ps, pc).pvalue
    rows.append(dict(dataset=n, all_skew=ROT[n]["jp"]["R2_Mskew_2D"], all_best=ROT[n]["jp"]["R2_Mbest_2D"],
                     straight_skew=rs["jp"]["R2_Mskew_2D"], straight_best=rs["jp"]["R2_Mbest_2D"],
                     curved_skew=rc["jp"]["R2_Mskew_2D"], curved_best=rc["jp"]["R2_Mbest_2D"],
                     signrank_p=p, straight_cond_median=np.median(ps), curved_cond_median=np.median(pc)))
JR = pd.DataFrame(rows).set_index("dataset")
# the pairing of the signed-rank test: condition 3j+1 (no barrier) with 3j+2 (barriers), the same target
for m in MONKEYS:
    dd = DS[f"M1-{m}"]
    gap = np.linalg.norm(dd["kin"][straight, -1] - dd["kin"][curved, -1], axis=1)
    print(f"monkey {m}: pairs {list(zip(dd['conds'][straight][:3], dd['conds'][curved][:3]))}...; hand position at "
          f"+850 ms, straight {np.round(dd['kin'][straight[:3], -1], 0).tolist()} and curved "
          f"{np.round(dd['kin'][curved[:3], -1], 0).tolist()} (mm); distance between the members of a pair "
          f"median {np.median(gap):.1f} mm, max {gap.max():.1f} mm, against {np.median(np.linalg.norm(dd['kin'][straight, -1], axis=1)):.0f} mm reach length")
print(JR.round(3).to_string())
for rd in ("skew", "best"):
    ok_all = JR[f"all_{rd}"].between(0.576, 0.803)
    ok_s = JR[f"straight_{rd}"].between(0.657, 0.968)
    ok_c = JR[f"curved_{rd}"].between(0.504, 0.77)
    print(f"reading {rd}: all in 0.576..0.803 {ok_all.sum()}/4, straight in 0.657..0.968 {ok_s.sum()}/4, "
          f"curved in 0.504..0.77 {ok_c.sum()}/4; straight > curved {int((JR[f'straight_{rd}'] > JR[f'curved_{rd}']).sum())}/4")

**Result: partially reproduced.** Under reading (b), the unconstrained fit, the R2 over all conditions is 0.831 (M1-J), 0.583 (PMd-J), 0.656 (M1-N) and 0.475 (PMd-N): two of four inside 0.576 to 0.803 (reading a: one of four). Straight-only fits (0.748 to 0.882) and curved-only fits (0.575 to 0.770) are inside their ranges in all four datasets, and straight is above curved in all four. The per-condition signed-rank test gives p < 0.001 only for M1-J (p = 0.44 to 0.61 elsewhere). The pairs it compares, conditions 3j+1 and 3j+2, end within 4.8 mm (J) and 5.1 mm (N) of each other in the median (at most 11.3 mm, for reaches of about 130 mm), so each pair is one target with and without barriers.


### jpca-variance-explained
> Activity within the jPCA rotational planes explained only 7 to 12% of the population variance in
> peri-movement firing rates.

In [ ]:
for n in NAMES:
    r = ROT[n]
    print(f"{n}: {PLANES[n]} plane(s); variance in the planes (code: reconstruction without the mean against "
          f"the activity) mean {r['ve'].mean()*100:.1f}% (s.d. {r['ve'].std(ddof=1)*100:.1f}); in 6.3..13.2%: "
          f"{0.063 <= r['ve'].mean() <= 0.132}; variant with the mean added back {r['ve_with_mean'].mean()*100:.1f}%")

**Result: reproduced.** Activity in the retained jPCA planes holds 9.1%, 10.6%, 10.8% and 6.8% of the variance (s.d. 5.3 to 8.5% across conditions), all inside 6.3 to 13.2%. With the cross-condition mean added back to the reconstruction it would be 71.6% to 80.9%.

### fig4d-jpca-variance-correlation
> Variance in the jPCA planes was only moderately correlated with the total rotational variance of a
> condition (rho 0.39 to 0.46; 66% of jPCA planes correlated at p < 0.05).

Two readings of the rotational variance. (a) Methods, Explaining variability in jPCA-identified rotations:
"prior to quantifying the variance or extracting temporal basis functions we soft-normalized and mean
subtracted activity identically to jPCA"; the condition averages are mean-subtracted across conditions and
scaled by the jPCA soft normalisation, the TBF is extracted again from them (same k), and the variance of the
rotational part per condition is taken. (b) Code corrTotalRotationsWithjPCA.m: the original basis, with the
loadings mean-subtracted and scaled by the soft normalisation. The Methods come first (reader, step 2);
(b) is a variant.


In [ ]:
NORM = {}
for n in NAMES:
    X, r = DS[n]["X"], ROT[n]
    tn = ldr.eig_transform((X - X.mean(0)) * r["sn"][None, :, None], K[n])
    inds = np.where(tn["freq"] > 0)[0]
    ok = ~np.isnan(tn["B"][:, 0])
    NORM[n] = dict(tb=tn, act=[tn["L"][c][:, inds] @ tn["B"][ok][:, inds].T for c in range(72)])
    print(f"{n}: basis of the normalised data, rotation frequencies {np.round(sorted(set(np.round(tn['freq'][inds], 2))), 2)} Hz "
          f"(original basis {np.round(sorted(set(np.round(TBF[n]['freq'][TBF[n]['freq'] > 0], 2))), 2)})")
def code_rot_act(n, c):
    tb, r = TBF[n], ROT[n]
    inds = np.where(tb["freq"] > 0)[0]
    ok = ~np.isnan(tb["B"][:, 0])
    meanL = tb["L"].mean(0)
    return r["sn"][:, None] * ((tb["L"][c][:, inds] - meanL[:, inds]) @ tb["B"][ok][:, inds].T)
F4D = []
for n in NAMES:
    r = ROT[n]
    rv = {"a": np.array([a.var(1, ddof=1).sum() for a in NORM[n]["act"]]),
          "b": np.array([np.nansum(np.var(code_rot_act(n, c), axis=1, ddof=1)) for c in range(72)])}
    for p in range(PLANES[n]):
        jv = np.nansum(np.nanvar(r["rot"][:, 2 * p:2 * p + 2], axis=2, ddof=1), axis=1)
        for basis in ("a", "b"):
            rr = stats.pearsonr(jv, rv[basis])
            F4D.append(dict(basis=basis, dataset=n, plane=p + 1, rho=rr[0], p=rr[1]))
F4D = pd.DataFrame(F4D)
print(F4D.round(4).to_string(index=False))
for basis, lab in (("a", "Methods: basis from the normalised data"), ("b", "code: original basis")):
    sub = F4D[F4D.basis == basis]
    share = (sub.p < 0.05).mean() * 100
    mean_rho = sub.groupby("dataset").rho.mean()
    sig = sub[sub.p < 0.05]
    print(f"{lab}: share of planes at p < 0.05 {share:.0f}% (paper 66, tolerance 59.4..72.6: {59.4 <= share <= 72.6}); "
          f"range reading (a), mean rho per dataset {mean_rho.round(3).to_dict()}, all in 0.351..0.506: "
          f"{mean_rho.between(0.351, 0.506).all()}; range reading (b), rho of the planes at p < 0.05 "
          f"{sig.rho.round(3).tolist()}, all in 0.351..0.506: {bool(len(sig)) and sig.rho.between(0.351, 0.506).all()}")


**Result: not reproduced.** Under both readings of the rotational variance, five of the six retained jPCA planes (83%) correlate with it at p < 0.05, outside 59.4 to 72.6%. With the basis extracted from the soft-normalised, mean-subtracted data (Methods) the correlations are -0.104 to 0.924 (per-dataset means 0.311 to 0.924; planes at p < 0.05, 0.370 to 0.924); with the code's original basis -0.088 to 0.919 (means 0.321 to 0.919). Neither reading of the range puts every value in 0.351 to 0.506.


### fig4f-plane-alignment-explains-jpca
> Conditions without clear rotations in the jPCA planes had rotations misaligned with those planes; the
> alignment index with the jPCA plane correlated strongly with the variance jPCA captured (rho 0.82 to 0.97).

The total rotational activity under the same two readings as fig4d: (a) the basis extracted from the
soft-normalised, mean-subtracted data (Methods), (b) the code's corrOverlapWithjPCA.m input, the original
basis with normalised loadings (variant).


In [ ]:
F4F = []
for n in NAMES:
    r = ROT[n]
    for p in range(PLANES[n]):
        J = r["jp"]["jPCs_highD"][:, 2 * p:2 * p + 2]
        jv = np.nansum(np.nanvar(r["rot"][:, 2 * p:2 * p + 2], axis=2, ddof=1), axis=1)
        for basis in ("a", "b"):
            acts = NORM[n]["act"] if basis == "a" else [code_rot_act(n, c) for c in range(72)]
            ov = np.array([ldr.alignment_index(a, J) for a in acts])
            rr = stats.pearsonr(jv, ov)
            F4F.append(dict(basis=basis, dataset=n, plane=p + 1, rho=rr[0], p=rr[1], align_min=ov.min(), align_max=ov.max()))
F4F = pd.DataFrame(F4F)
print(F4F.round(4).to_string(index=False))
for basis in ("a", "b"):
    sub = F4F[F4F.basis == basis]
    okf = (sub.rho.between(0.738, 1)) & (sub.p < 0.001)
    print(f"reading {basis}: planes with rho in 0.738..1 and p < 0.001: {okf.sum()} of {len(sub)}")


**Result: partially reproduced.** With the basis extracted from the soft-normalised, mean-subtracted data (Methods), the alignment of a condition's rotations with a jPCA plane correlates with the variance in that plane with rho 0.843 to 0.952 (p < 0.001) for four of the six planes, and 0.660 (first M1-N plane) and 0.475 (PMd-N) for the other two. With the code's input (original basis, normalised loadings) five of six are in 0.738 to 1 (0.826 to 0.947) and the first M1-N plane gives 0.685. At least half hold under both.


## single-lds-variance
> A single linear dynamical system fitted to all conditions captured on average 46 to 66% of the population
> variance.

Code fitSingleLDS.m: one full-rank M from each bin to the next over all conditions from bin 20 on; each
condition simulated forward from bin 20.

In [ ]:
SINGLE = {}
for n in NAMES:
    X = DS[n]["X"]
    X1 = np.concatenate([x[:, ldr.T0:-1] for x in X], 1)
    X2 = np.concatenate([x[:, ldr.T0 + 1:] for x in X], 1)
    M = X2 @ np.linalg.pinv(X1)
    va, vb = [], []
    for x in X:
        s = x.copy()
        for t in range(ldr.T0 + 1, ldr.NT):
            s[:, t] = M @ s[:, t - 1]
        sa = s.copy(); sa[:, :ldr.T0] = np.nan
        va.append(ldr.var_explained(x[None], sa[None], clip=False)[0])
        vb.append(1 - (s - x).var(1, ddof=1).sum() / x.var(1, ddof=1).sum())
    va, vb = np.array(va), np.array(vb)
    SINGLE[n] = va
    print(f"{n}: (a) -150..850 mean {va.mean()*100:.1f}% (s.d. {va.std(ddof=1)*100:.1f}, min {va.min()*100:.1f}); "
          f"(b) whole window {vb.mean()*100:.1f}%; in 41.4..72.6%: {0.414 <= va.mean() <= 0.726}, {0.414 <= vb.mean() <= 0.726}")

**Result: not reproduced.** A single LDS over all conditions explains 77.3%, 81.3%, 82.6% and 78.6% from -150 ms on (79.6% to 86.5% over the whole window), above 41.4 to 72.6% in every dataset under both windows.

## fig2c-condition-lds-variance
> Activity on each single condition was low-dimensional and approximated by an LDS of 5 to 9 dimensions,
> which explained 71 to 93% of the population variance.

The LDS per condition is the code's idDynamics.m (reduced-rank regression of the least-squares dynamics
from bin 20 on, simulation from bin 20, half-lives capped at 10 s). Dimensionality: the Methods (Fitting
dynamics to single conditions) describe the cross-validation as projecting one partition's average onto
its top k singular vectors and measuring the variance explained in the other's; the code's
crossValidateDimensionalityConditions.m scores the LDS fit itself (its idDynamics returns the training
data as the approximation, so the published code cannot choose k; the simulation is used here). Both
scores are run, with odd k from 1 to 21 (code), under the Methods' even split and the code's 3/4 split.

In [ ]:
KS = np.arange(1, 22, 2)
def lds_cv(n, seed_k, folds=100, ratio=0.5):
    d = DS[n]
    g = ldr.rng("fig2c-condition-lds-variance", seed_k)
    proj = np.zeros((len(KS), folds, 72)); sim = np.zeros((len(KS), folds, 72))
    for f in range(folds):
        a = ldr.split_halves(d["cond"], g, ratio)
        Xa = ldr.cond_average(d["Xt"][a], d["cond"][a]); Xb = ldr.cond_average(d["Xt"][~a], d["cond"][~a])
        W, Yb = Xa[:, :, ldr.T0:], Xb[:, :, ldr.T0:]
        den = Yb.var(2, ddof=1).sum(1)
        Up = np.linalg.svd(W, full_matrices=False)[0]
        o = ldr.lds_batch(Xa, KS)
        for i, k in enumerate(KS):
            P = Up[:, :, :k]
            rec = P @ (np.transpose(P, (0, 2, 1)) @ W)
            proj[i, f] = np.clip(1 - (Yb - rec).var(2, ddof=1).sum(1) / den, 0, None)
            sm_ = o[k][1][:, :, ldr.T0:]
            sim[i, f] = np.clip(1 - (Yb - sm_).var(2, ddof=1).sum(1) / den, 0, None)
    return proj.mean((1, 2)), sim.mean((1, 2))

T = time.time()
jobs = [(n, 0, 100, 0.5) for n in NAMES] + [(n, 1, 20, 0.75) for n in NAMES] + \
       [(n, k, 10, 0.5) for n in NAMES for k in SEED_OFFSETS[1:]]
LOUT = dict(zip(jobs, pmap(lds_cv, jobs)))
LCV = {n: {0.5: LOUT[(n, 0, 100, 0.5)], 0.75: LOUT[(n, 1, 20, 0.75)]} for n in NAMES}
print(f"cross-validation in {time.time() - T:.0f} s")

In [ ]:
KLDS = {}
LDSVE = {}
for n in NAMES:
    X = DS[n]["X"]
    line = []
    for ratio in (0.5, 0.75):
        proj, sim = LCV[n][ratio]
        kp, ks = KS[proj.argmax()], KS[sim.argmax()]
        line.append(f"split {ratio}: projection k={kp} (curve {np.round(proj[:6], 3)}), "
                    f"LDS k={ks} (curve {np.round(sim[:6], 3)})")
        if ratio == 0.5:
            KLDS[n] = dict(proj=kp, sim=ks)
    print(n + ": " + "; ".join(line))
    ks_ = sorted(set([KLDS[n]["proj"], KLDS[n]["sim"], 5, 7, 9]))
    fits = ldr.lds_batch(X, ks_)
    for kk in ks_:
        ve = ldr.var_explained(X, fits[kk][1])
        LDSVE[(n, kk)] = ve
        print(f"     k={kk}: mean VE {ve.mean()*100:.1f}% (s.d. {ve.std(ddof=1)*100:.1f})")
for n in NAMES:
    for rd in ("proj", "sim"):
        kk = KLDS[n][rd]
        print(f"{n} reading {rd}: k={kk} in 5..9: {5 <= kk <= 9}; VE {LDSVE[(n, kk)].mean()*100:.1f}% in 63.9..100: "
              f"{LDSVE[(n, kk)].mean() >= 0.639}")

**Result: reproduced.** Cross-validation chooses 5 dimensions for the J arrays and 7 for the N arrays under the Methods' projection score (the LDS score gives 5 for PMd-N), all inside 5 to 9, and the single-condition LDSs explain 83.4%, 90.7%, 92.9% and 94.4% (91.0% for PMd-N at 5), all above 63.9%. The choice is the same under the four other seeds.

The code's 3/4 split above runs 20 folds, for time. Seed check: the chosen k under the other four seeds,
with 10 folds each, for time (the claim's own seed above uses the full 100).

In [ ]:
for n in NAMES:
    print(n, "k (projection, LDS) under seeds +1000..+4000, 20 folds:",
          [(int(KS[LOUT[(n, k, 10, 0.5)][0].argmax()]), int(KS[LOUT[(n, k, 10, 0.5)][1].argmax()])) for k in SEED_OFFSETS[1:]])

## fig3a-eigenvalue-clusters
> Eigenvalues of the single-condition LDSs formed distinct clusters, corresponding to rotations at about
> 0.5, 1.5, 2.5 and 4 Hz on every condition, along with an offset.

In [ ]:
EIG = {}
for n in NAMES:
    X = DS[n]["X"]
    for rd in ("proj", "sim"):
        kk = KLDS[n][rd]
        E = ldr.lds_eigs_batch(X, kk)
        EIG[(n, rd)] = E
        npair = (kk - 1) // 2
        fr = []
        for c in range(72):
            cf = np.sort(ldr.freq_hz(E[c][np.imag(E[c]) > 1e-12]))
            fr.append(np.r_[cf, np.full(npair, np.nan)][:npair])
        fr = np.array(fr)
        med = np.nanmedian(fr, 0)
        nreal = (np.abs(np.imag(E)) <= 1e-12).sum(1)
        tgt = np.array([0.5, 1.5, 2.5, 4.0])[:npair]
        ok = np.abs(med - tgt) <= 0.1 * tgt
        print(f"{n} reading {rd} (k={kk}): median pair frequencies {np.round(med, 2)} Hz against {tgt}; within 10%: "
              f"{ok}; conditions with a real eigenvalue {np.mean(nreal >= 1)*100:.0f}%; frequency IQR per pair "
              f"{np.round(np.nanpercentile(fr, 75, 0) - np.nanpercentile(fr, 25, 0), 2)}")

**Result: not reproduced.** Every condition has a real eigenvalue, but the median frequencies of the eigenvalue pairs are 0.85 and 2.38 Hz (M1-J), 0.57 and 1.51 Hz (PMd-J), 0.63, 1.75 and 3.19 Hz (M1-N) and 0.62, 1.37 and 2.54 Hz (PMd-N): 3 of the 10 are within 10% of 0.5, 1.5 and 2.5 Hz in order.

## fig3c-eigenvalue-distance-auc
> Eigenvalues differed between conditions only slightly more than expected from estimation noise, with
> ROC-AUC 0.51 to 0.58 between the between-condition distances and the noise distribution.

Distance: mean over corresponding (frequency-sorted) eigenvalues of their distance (main), and the code's
norm of the difference (variant). Nulls: (b) random halves of the real trials, run with 150 repeats per condition (the Methods and the code
compareTrialSimilarity.m use 500); (a) two sets of 20 Poisson trials from each condition's rates, smoothed and
averaged, run with 30 repeats per condition (the Methods' 500 did not fit the time).
The LDS dimensionality is the one chosen above (projection reading).

In [ ]:
def eig_dist(E1, E2, kind="mean"):
    d = np.abs(E1 - E2)
    return d.mean(-1) if kind == "mean" else np.sqrt((d ** 2).sum(-1))

def null_halves(n, kk, k, reps):
    d = DS[n]
    g = ldr.rng("fig3c-eigenvalue-distance-auc", k)
    out = np.zeros((2, 72, reps, kk), complex)
    for r in range(reps):
        a = ldr.split_halves(d["cond"], g, 0.5)
        out[0, :, r] = ldr.lds_eigs_batch(ldr.cond_average(d["Xt"][a], d["cond"][a]), kk)
        out[1, :, r] = ldr.lds_eigs_batch(ldr.cond_average(d["Xt"][~a], d["cond"][~a]), kk)
    return out

def smooth_matrix():
    """The code's smoothing and down-sampling as a 1201 x 121 matrix."""
    e = np.zeros((1201, 1210), np.float32)
    e[np.arange(1201), 5 + np.arange(1201)] = 1
    return ldr.smooth(e / 1000)

def null_poisson(n, kk, k, reps, ntr=20):
    """Two sets of ntr Poisson trials per condition from its rates (linear interpolation of the 10 ms
    average to 1 ms, code generateTrials.m), smoothed and averaged; the mean of ntr trials is drawn as one
    Poisson count with ntr times the rate, divided by ntr."""
    g = ldr.rng("fig3c-eigenvalue-distance-auc", k)
    KERN = smooth_matrix()
    X = DS[n]["X"]
    tt = np.arange(-350, 851)
    lam = np.stack([[np.interp(tt, ldr.TIMES, x) for x in Xc] for Xc in X]) / 1000 * ntr
    lam = np.clip(lam, 0, None)
    out = np.zeros((2, 72, reps, kk), complex)
    for r in range(reps):
        for h in range(2):
            xa = (g.poisson(lam).astype(np.float32) / ntr) @ KERN
            out[h, :, r] = ldr.lds_eigs_batch(xa.astype(float), kk)
    return out

T = time.time()
jobs_b = [(n, KLDS[n]["proj"], 0, 150) for n in NAMES]
jobs_a = [(n, KLDS[n]["proj"], 1, 30) for n in NAMES]
jobs_s = [(n, KLDS[n]["proj"], k, 20) for n in NAMES for k in SEED_OFFSETS[1:]]
NB = dict(zip([j[0] for j in jobs_b], pmap(null_halves, jobs_b)))
NA = dict(zip([j[0] for j in jobs_a], pmap(null_poisson, jobs_a)))
NS = dict(zip([(j[0], j[2]) for j in jobs_s], pmap(null_halves, jobs_s)))
F3C = {}
for n in NAMES:
    E = EIG[(n, "proj")]
    obs = {kind: np.array([eig_dist(E[i], E[j], kind) for i in range(72) for j in range(72) if i != j]) for kind in ("mean", "norm")}
    F3C[n] = dict(obs=obs, nb=NB[n], na=NA[n], k=KLDS[n]["proj"])
print(f"nulls in {time.time() - T:.0f} s")

In [ ]:
for n in NAMES:
    f = F3C[n]
    line = []
    for nm, nl in (("halves", f["nb"]), ("Poisson", f["na"])):
        for kind in ("mean", "norm"):
            nd = eig_dist(nl[0], nl[1], kind).ravel()
            auc, raw = ldr.auc_folded(f["obs"][kind], nd)
            line.append(f"{nm}/{kind}: AUC {auc:.3f} (obs median {np.median(f['obs'][kind]):.4f} vs null {np.median(nd):.4f})")
    print(f"{n} (k={f['k']}): " + "; ".join(line))

**Result: not reproduced.** With the mean distance, the AUC against the halves null is 0.717, 0.669, 0.660 and 0.580, and against the Poisson null 0.797, 0.729, 0.646 and 0.592; under either null only PMd-N lies in 0.459 to 0.638. The code's norm distance gives the same picture (0.577 to 0.787). The halves AUC is stable over the other four seeds (0.577 to 0.717).

Seed check for the halves null with the mean distance (20 repeats for each of the other four seeds, for time).

In [ ]:
for n in NAMES:
    print(n, "AUC under seeds +1000..+4000 (20 repeats):",
          np.round([ldr.auc_folded(F3C[n]["obs"]["mean"], eig_dist(NS[(n, k)][0], NS[(n, k)][1], "mean").ravel())[0]
                    for k in SEED_OFFSETS[1:]], 3))

## suppfig2-eigenvalues-from-kinematics
> Kinematic parameters predicted the eigenvalues poorly (mean R2 = 0.1, leave-one-out cross-validation).

Code getCorrelationsWithFrequency.m: predictors the four reach parameters (kinematic basis as in fig6e,
both readings), the reach duration (as in fig3i) and a constant; targets the real and imaginary parts of
each eigenvalue of the condition's LDS; least squares, leave one condition out; R2 per component, negatives
0, components with no variance (the imaginary part of a real eigenvalue) left out; mean over components
pooled over the four datasets. Both members of a conjugate pair are targets, as the criterion says ("the real and imaginary parts of
each eigenvalue"), so each real part counts twice; the variant counts each pair once.


In [ ]:
KB = {rd: ldr.kin_basis([DS[f"M1-{m}"]["kin"] for m in MONKEYS], rd) for rd in ("svd", "skewnormal")}
for rd in ("svd", "skewnormal"):
    for erd in ("proj", "sim"):
        for once in (False, True):
            r2s = []
            per = {}
            for n in NAMES:
                E = EIG[(n, erd)]
                if once:   # drop a column that is the conjugate of the one before it in most conditions
                    dup = [j > 0 and np.mean(np.isclose(E[:, j], np.conj(E[:, j - 1])) & (np.abs(np.imag(E[:, j])) > 1e-12)) > 0.5
                           for j in range(E.shape[1])]
                    E = E[:, ~np.array(dup)]
                Y = np.c_[np.real(E), np.imag(E)]
                Y = Y[:, Y.std(0) > 1e-12]
                P = ldr.reach_params(KB[rd][0], DS[n]["kin"])
                Xk = np.c_[P, WARP[(n, "trial")]["dur"], np.ones(72)]
                Yh = np.zeros_like(Y)
                for c in range(72):
                    tr = np.arange(72) != c
                    W = np.linalg.pinv(Xk[tr]) @ Y[tr]
                    Yh[c] = Xk[c] @ W
                r2 = np.clip(1 - (Yh - Y).var(0, ddof=1) / Y.var(0, ddof=1), 0, None)
                per[n] = r2.mean(); r2s.extend(r2)
            print(f"kinematic basis {rd}, eigenvalues {erd}, {'each pair once' if once else 'both members (criterion)'}: "
                  f"mean R2 pooled {np.mean(r2s):.3f} (in 0.05..0.15: {0.05 <= np.mean(r2s) <= 0.15}); per dataset "
                  f"{({k: round(v, 3) for k, v in per.items()})}")


**Result: not reproduced.** The reach parameters and duration (each trial's own, as fig3i reading a) predict the eigenvalue components with a pooled mean leave-one-out R2 of 0.243 (projection-chosen dimensionality) or 0.295 (LDS-chosen) under the svd basis (0.243 and 0.296 under the skew-normal basis), outside 0.05 to 0.15; per dataset it is 0.149 to 0.476. Counting each conjugate pair once gives 0.235 and 0.291.


## suppfig3-model-comparison
> The model with shared eigenvalues and condition-specific planes (LDR) explained more cross-validated
> variance than jPCA or condition-specific LDSs.

The criterion's main reading compares the held-out TBF with the in-sample LDS and jPCA, and the all
in-sample comparison is its variant. The legend of Supplementary Figure 3 says that the LDR model "explains
greater neural variance when cross-validated" than the condition-specific LDS; a like-for-like variant, asked
by the verifier, scores both on the held-out half of an even split over the same window (-160 ms on, where
the LDS is simulated): the TBF basis and loadings from one half, and the LDS's held-out score from the
cross-validation of fig2c at the chosen k. This variant is shown; it does not decide the outcome (the
question is put to a reader, questions/sabatini-2024.yaml).


In [ ]:
def tbf_heldout(n, k, g, folds=100):
    """Held-out variance explained of the TBF at k from -160 ms on: basis and loadings from one half of
    each condition's trials, scored on the other half's average."""
    d = DS[n]
    out = np.zeros(folds)
    for f in range(folds):
        a = ldr.split_halves(d["cond"], g, 0.5)
        Xa = ldr.cond_average(d["Xt"][a], d["cond"][a]); Xb = ldr.cond_average(d["Xt"][~a], d["cond"][~a])
        C, N, T = Xa.shape
        B = np.linalg.svd(Xa.reshape(C * N, T), full_matrices=False)[2][:k].T
        rec = (Xa @ B) @ B.T
        rec[:, :, :ldr.T0] = np.nan
        out[f] = ldr.var_explained(Xb, rec).mean()
    return out.mean()

rows = []
for n in NAMES:
    cv_tbf = CV[n]["mean"][K[n] - 1]
    lds_ve = LDSVE[(n, KLDS[n]["proj"])].mean()
    jp = ROT[n]["ve"].mean()
    tbf_w = tbf_heldout(n, K[n], ldr.rng("suppfig3-model-comparison"))
    sim_curve = LCV[n][0.5][1]
    lds_ho = sim_curve[list(KS).index(KLDS[n]["proj"])]
    rows.append(dict(dataset=n, tbf_heldout=cv_tbf, tbf_insample=VE_TBF[n].mean(), lds_insample=lds_ve,
                     lds_k=KLDS[n]["proj"], jpca=jp, tbf_heldout_from_m160=tbf_w, lds_heldout_from_m160=lds_ho,
                     lds_heldout_best_k=sim_curve.max()))
    print(f"{n}: TBF held-out {cv_tbf*100:.1f}% (in-sample {VE_TBF[n].mean()*100:.1f}%), single-condition LDS "
          f"{lds_ve*100:.1f}% (k={KLDS[n]['proj']}), jPCA {jp*100:.1f}%; TBF highest (cross-validated, main): "
          f"{cv_tbf > max(lds_ve, jp)}; (in-sample): {VE_TBF[n].mean() > max(lds_ve, jp)}")
    print(f"     held-out from -160 ms on: TBF {tbf_w*100:.1f}%, LDS {lds_ho*100:.1f}% at k={KLDS[n]['proj']} "
          f"(best k of the LDS score {sim_curve.max()*100:.1f}%); TBF higher: {tbf_w > lds_ho}")
M3 = pd.DataFrame(rows).set_index("dataset")
print(f"main reading, TBF highest in {(M3.tbf_heldout > M3[['lds_insample', 'jpca']].max(axis=1)).sum()}/4; "
      f"in-sample variant {(M3.tbf_insample > M3[['lds_insample', 'jpca']].max(axis=1)).sum()}/4; held-out variant (TBF above "
      f"held-out LDS and jPCA) {((M3.tbf_heldout_from_m160 > M3.lds_heldout_from_m160) & (M3.tbf_heldout_from_m160 > M3.jpca)).sum()}/4")


**Result: not reproduced.** Under the criterion's main reading the cross-validated TBF variance (73.5%, 81.0%, 88.8%, 81.7%) is below the in-sample single-condition LDS (83.4%, 90.7%, 92.9%, 94.4%) in every dataset, though above jPCA (6.8% to 10.8%). In the all in-sample variant the TBF (92.3% to 97.1%) is the highest in all four. In the held-out variant over the same window (from -160 ms), the TBF is above the held-out LDS and jPCA in all four datasets (table above); this reading of the legend is put to a reader as a question and does not decide the outcome.


## Geometry of the rotations
Alignment index as in the code (getAlignmentIndex.m). Noise: per condition, loadings regressed on the
fixed (corrected raw) basis for two random halves of its trials, 36 repeats (Methods; the code's 200 a
variant, not run for time). ROC-AUC folded to at least 0.5.

In [ ]:
def noise_loadings(d, tb, g, reps):
    raw = tb["raw"]
    pin = np.linalg.pinv(raw.T)
    L1 = np.zeros((reps, 72) + tb["L"].shape[1:]); L2 = np.zeros_like(L1)
    for r in range(reps):
        a = ldr.split_halves(d["cond"], g, 0.5)
        L1[r] = ldr.cond_average(d["Xt"][a], d["cond"][a]) @ pin
        L2[r] = ldr.cond_average(d["Xt"][~a], d["cond"][~a]) @ pin
    return L1, L2

def ai_matrix(L, B, inds, inds2=None):
    inds2 = inds if inds2 is None else inds2
    ok = ~np.isnan(B[:, 0])
    out = np.zeros((72, 72))
    for i in range(72):
        D = L[i][:, inds] @ B[ok][:, inds].T
        Dc = D - D.mean(1, keepdims=True)
        den = (Dc ** 2).sum()
        for j in range(72):
            Q = np.linalg.qr(L[j][:, inds2])[0]
            r = Dc - Q @ (Q.T @ Dc)
            out[i, j] = max(0, 1 - (r ** 2).sum() / den)
    return out

GEO = {}
T = time.time()
NOISE = {(n, k): noise_loadings(DS[n], TBF[n], ldr.rng("fig4b-plane-alignment", k), 36)
         for n in NAMES for k in SEED_OFFSETS}
for n in NAMES:
    tb = TBF[n]
    gr = ldr.groups(tb)
    L1, L2 = NOISE[(n, 0)]
    ok = ~np.isnan(tb["B"][:, 0])
    comps = {}
    for f, inds in gr:
        obs = ai_matrix(tb["L"], tb["B"], inds)
        noise = np.array([[ldr.alignment_index(L1[r, c][:, inds] @ tb["B"][ok][:, inds].T, L2[r, c][:, inds])
                           for r in range(36)] for c in range(72)])
        comps[f] = dict(inds=inds, obs=obs, noise=noise)
    GEO[n] = dict(groups=gr, comps=comps, L1=L1, L2=L2)
print(f"alignment in {time.time() - T:.0f} s")

def offdiag(M):
    return M[~np.eye(len(M), dtype=bool)]

### fig4b-plane-alignment
> Same-frequency rotational planes on pairs of conditions were far less aligned than expected from
> estimation noise (ROC-AUC 0.85 to 0.96).

In [ ]:
rows = []
for n in NAMES:
    for f, cc in GEO[n]["comps"].items():
        if len(cc["inds"]) != 2:
            continue
        o, z = offdiag(cc["obs"]), cc["noise"].ravel()
        auc, _ = ldr.auc_folded(o, z)
        per_c = np.array([np.r_[cc["obs"][c, :c], cc["obs"][c, c + 1:]].mean() for c in range(72)])
        pa = stats.wilcoxon(per_c, cc["noise"].mean(1)).pvalue
        pb = stats.ranksums(o, z).pvalue
        rows.append(dict(dataset=n, freq=round(f, 2), obs_mean=o.mean(), noise_mean=z.mean(), auc=auc,
                         p_signrank=pa, p_ranksum=pb))
F4B = pd.DataFrame(rows)
print(F4B.round(4).to_string(index=False))
ok = F4B.auc.between(0.765, 1) & (F4B.obs_mean < F4B.noise_mean)
print(f"rotations with AUC in 0.765..1 and observed lower: {ok.sum()} of {len(F4B)}; signed-rank p < 0.001 for "
      f"{(F4B.p_signrank < 0.001).sum()}, rank-sum p < 0.001 for {(F4B.p_ranksum < 0.001).sum()}")
# seed check
for n in NAMES:
    tb = TBF[n]; ok_ = ~np.isnan(tb["B"][:, 0]); aucs = []
    for k in SEED_OFFSETS[1:]:
        A1, A2 = NOISE[(n, k)]
        a_ = []
        for f, cc in GEO[n]["comps"].items():
            if len(cc["inds"]) != 2: continue
            nz = np.array([[ldr.alignment_index(A1[r, c][:, cc["inds"]] @ tb["B"][ok_][:, cc["inds"]].T, A2[r, c][:, cc["inds"]])
                            for r in range(36)] for c in range(72)])
            a_.append(ldr.auc_folded(offdiag(cc["obs"]), nz.ravel())[0])
        aucs.append(np.round(a_, 3))
    print(n, "AUC per rotation under seeds +1000..+4000:", aucs)

**Result: partially reproduced.** For 13 of the 14 rotations the same-frequency planes of different conditions are less aligned than the half-split noise, with AUC 0.828 to 0.943 and signed-rank p < 0.001; the 3.65 Hz rotation of PMd-N gives AUC 0.545 (signed-rank p = 0.0034). The AUCs barely move under the other four seeds.

### suppfig6a-location-alignment
> The state space location occupied different dimensions across conditions, less aligned than expected
> from estimation noise (ROC-AUC 0.88 to 0.95).

In [ ]:
for n in NAMES:
    for f, cc in GEO[n]["comps"].items():
        if len(cc["inds"]) != 1:
            continue
        o, z = offdiag(cc["obs"]), cc["noise"].ravel()
        auc, _ = ldr.auc_folded(o, z)
        per_c = np.array([np.r_[cc["obs"][c, :c], cc["obs"][c, c + 1:]].mean() for c in range(72)])
        pa = stats.wilcoxon(per_c, cc["noise"].mean(1)).pvalue
        print(f"{n} offset (freq {f:.2f}): observed {o.mean():.3f}, noise {z.mean():.3f}, AUC {auc:.3f} (in 0.792..1: "
              f"{0.792 <= auc <= 1}), signed-rank p = {pa:.2g}, rank-sum p = {stats.ranksums(o, z).pvalue:.2g}")

**Result: partially reproduced.** The location is less aligned across conditions than the noise in every dataset (signed-rank p below 1e-11), with AUC 0.890 (M1-J) and 0.889 (M1-N) inside 0.792 to 1, but 0.698 (PMd-J) and 0.764 (PMd-N) below it.

### fig5a-frequencies-segregated and suppfig6c-location-distinct-from-rotations
> Different-frequency rotational planes on pairs of conditions were nearly orthogonal, and same-frequency
> planes were better aligned.

> State space location dimensions were distinct from those of the rotations.

In [ ]:
CROSS = {}
for n in NAMES:
    tb = TBF[n]
    gr = GEO[n]["groups"]
    for f1, i1 in gr:
        for f2, i2 in gr:
            if f1 == f2:
                continue
            CROSS[(n, f1, f2)] = ai_matrix(tb["L"], tb["B"], i1, i2)

In [ ]:
rows = []
for n in NAMES:
    gr = GEO[n]["groups"]
    for f1, i1 in gr:
        if len(i1) != 2: continue
        same = GEO[n]["comps"][f1]["obs"]
        for f2, i2 in gr:
            if f2 == f1 or len(i2) != 2: continue
            diff = CROSS[(n, f1, f2)]
            a, b = offdiag(same), offdiag(diff)
            pa = stats.wilcoxon(a, b).pvalue
            pb = stats.ranksums(a, b).pvalue
            rows.append(dict(dataset=n, rot=round(f1, 2), other=round(f2, 2), same=a.mean(), different=b.mean(),
                             p_paired=pa, p_ranksum=pb))
F5A = pd.DataFrame(rows)
print(F5A.round(4).to_string(index=False))
for col in ("p_paired", "p_ranksum"):
    okk = (F5A.same > F5A.different) & (F5A[col] < 0.001)
    print(f"{col}: same higher with p < 0.001 in {okk.sum()} of {len(F5A)} ({okk.mean()*100:.0f}%)")

**Result: partially reproduced.** Same-frequency alignment is higher than different-frequency alignment with p < 0.001 in 35 of 36 comparisons under the paired reading (97%; the exception is PMd-J 2.64 against 1.64 Hz, 0.202 against 0.202) and 33 of 36 under the rank-sum reading.

In [ ]:
for n in NAMES:
    gr = GEO[n]["groups"]
    off = [f for f, i in gr if len(i) == 1][0]
    same = offdiag(GEO[n]["comps"][off]["obs"])
    for f2, i2 in gr:
        if f2 == off: continue
        b = offdiag(CROSS[(n, off, f2)])
        print(f"{n}: location->location {same.mean():.3f} vs location->rotation {f2:.2f} Hz {b.mean():.3f}; "
              f"paired signed-rank p = {stats.wilcoxon(same, b).pvalue:.2g}, rank-sum p = {stats.ranksums(same, b).pvalue:.2g}")

**Result: reproduced.** In every dataset the location is far better aligned with other conditions' locations (0.892 to 0.969) than with any rotation's plane (0.104 to 0.580), with p printed as 0 by both tests.

### fig4c-subspace-excursion-angles and suppfig6b-location-dimensionality (SEA part)
Subspace excursion angles to 25 dimensions (code subspaceExcursionAngles.m, maxTol 0). Noise: per condition,
the SEA of the 72 half-split loading estimates (36 repeats x 2 halves; the code uses 200 repeats, 400
estimates), and for each dimension the fraction of conditions whose noise angle exceeds the data's
(code singleFrequencyOverlapControl.m).

In [ ]:
def sea_job(n, gi):
    tb = TBF[n]
    f, inds = GEO[n]["groups"][gi]
    L1, L2 = GEO[n]["L1"], GEO[n]["L2"]
    data_sea = ldr.sea(tb["L"][:, :, inds], 25)
    noise = np.array([ldr.sea(np.concatenate([L1[:, c][:, :, inds], L2[:, c][:, :, inds]]), 25) for c in range(72)])
    m = min(noise.shape[1], len(data_sea))
    return f, dict(data=data_sea, noise=noise, p=(noise[:, :m] > data_sea[None, :m]).mean(0))

T = time.time()
jobs = [(n, gi) for n in NAMES for gi in range(len(GEO[n]["groups"]))]
SEA = {(j[0], r[0]): r[1] for j, r in zip(jobs, pmap(sea_job, jobs))}
print(f"SEA in {time.time() - T:.0f} s")

In [ ]:
rows = []
for (n, f), s in SEA.items():
    if not any(ff == f and len(i) == 2 for ff, i in GEO[n]["groups"]):
        continue
    n45 = int((s["data"] > 45).sum())
    nabove = int((s["p"] < 0.05).sum())
    rows.append(dict(dataset=n, freq=round(f, 2), above45=n45, above_noise=nabove,
                     first=s["data"][0], tenth=s["data"][9], last=s["data"][-1]))
F4C = pd.DataFrame(rows)
print(F4C.round(1).to_string(index=False))
print("counts above 45 deg in 5.4..22:", F4C.above45.between(5.4, 22).sum(), "of", len(F4C),
      "| dimensions above noise exceed the count above 45 deg:", (F4C.above_noise > F4C.above45).sum(), "of", len(F4C))

**Result: partially reproduced.** The number of excursion angles above 45 degrees is 7 to 21 for the J and M1-N rotations, inside 5.4 to 22, but 23 to 25 for PMd-N (10 of 14 in range); the data's angles exceed the half-split noise at all 25 dimensions, more than the count above 45 degrees in 11 of 14 rotations (the PMd-N rotations with 25 are equal).

### rotation-orientation-dimensionality and suppfig6b-location-dimensionality
> Rotation orientations required 23 to 40 principal components to capture 80% of their variance, had 7 to
> 33 dimensions with SNR above 1, and participation ratios of 16.7 to 36.8.

PCA readings: (a) the loading columns of each condition stacked as observations; (b) the rotation's
activity (loading columns times basis functions) of all conditions as observations. SNR dimensionality:
code crossValidateDimensionalityRotations.m (halves, 100 folds; basis from one half, loadings of both
halves on its raw basis; the dimensionality of the PCA of half 1's loadings that maximises the variance
explained in half 2's rotation, 1 to 50 dimensions).

In [ ]:
def snr_dims(d, k, g, folds=100):
    res = {}
    for fo in range(folds):
        a = ldr.split_halves(d["cond"], g, 0.5)
        Xa = ldr.cond_average(d["Xt"][a], d["cond"][a]); Xb = ldr.cond_average(d["Xt"][~a], d["cond"][~a])
        tb = ldr.eig_transform(Xa, k)
        Lb = Xb @ np.linalg.pinv(tb["raw"].T)
        Bv = tb["B"][ldr.T0 + 1:]
        for gi, (f, inds) in enumerate(ldr.groups(tb)):
            cols = np.concatenate(list(tb["L"][:, :, inds]), 1)
            U = np.linalg.svd(cols, full_matrices=False)[0]
            tr = Lb[:, :, inds] @ Bv[:, inds].T      # 72 x N x T
            den = tr.var(2, ddof=1).sum(1)
            for dim in range(1, min(51, U.shape[1] + 1)):
                P = U[:, :dim] @ U[:, :dim].T
                pr = np.einsum("nm,cmk->cnk", P, tb["L"][:, :, inds]) @ Bv[:, inds].T
                ve = 1 - (pr - tr).var(2, ddof=1).sum(1) / den
                res.setdefault((gi, len(inds)), np.zeros((folds, 50)))[fo, dim - 1] = ve.mean()
    return {key: int(v.mean(0).argmax() + 1) for key, v in res.items()}

T = time.time()
def snr_job(n):
    return snr_dims(DS[n], K[n], ldr.rng("rotation-orientation-dimensionality"), 100)
SNR = dict(zip(NAMES, pmap(snr_job, [(n,) for n in NAMES])))
print(f"SNR dimensionality in {time.time() - T:.0f} s")

In [ ]:
rows = []
for n in NAMES:
    tb = TBF[n]
    ok = ~np.isnan(tb["B"][:, 0])
    for gi, (f, inds) in enumerate(GEO[n]["groups"]):
        colsA = np.concatenate([tb["L"][c][:, inds].T for c in range(72)])          # (72*w) x N
        actB = np.concatenate([(tb["L"][c][:, inds] @ tb["B"][ok][:, inds].T).T for c in range(72)])
        pa, pra = ldr.pca_dims(colsA)
        pb, prb = ldr.pca_dims(actB)
        rows.append(dict(dataset=n, freq=round(f, 2), kind="rotation" if len(inds) == 2 else "location",
                         pcs80_a=pa, pr_a=pra, pcs80_b=pb, pr_b=prb, snr_dims=SNR[n].get((gi, len(inds)))))
DIM = pd.DataFrame(rows)
print(DIM.round(1).to_string(index=False))
R = DIM[DIM.kind == "rotation"]
for rd in ("a", "b"):
    print(f"rotations, reading {rd}: PCs to 80% in 20.7..44 {R[f'pcs80_{rd}'].between(20.7, 44).sum()}/{len(R)}; "
          f"participation ratio in 15.03..40.48 {R[f'pr_{rd}'].between(15.03, 40.48).sum()}/{len(R)}")
print(f"rotations: SNR dimensions in 6.3..36.3 {R.snr_dims.between(6.3, 36.3).sum()}/{len(R)}")

**Result: not reproduced.** The rotations need 3 to 24 PCs for 80% of their variance (reading a; 3 to 22 for reading b) and have participation ratios of 2.3 to 19.6, so 3 of 14 and 2 of 14 rotations are in 20.7 to 44 and 15.03 to 40.48. The SNR dimensionality (5 to 19) is in 6.3 to 36.3 for 13 of 14.

In [ ]:
Lc = DIM[DIM.kind == "location"]
for n in NAMES:
    off = [f for f, i in GEO[n]["groups"] if len(i) == 1][0]
    s = SEA[(n, off)]
    n15 = int((s["data"] > 15).sum())
    print(f"{n} location: SEA angles above 15 deg {n15} (in 3.6..5.5: {3.6 <= n15 <= 5.5}); first angles "
          f"{np.round(s['data'][:8], 1)}; dimensions above noise {int((s['p'] < 0.05).sum())}")
print(Lc.round(1).to_string(index=False))
for rd in ("a", "b"):
    print(f"location, reading {rd}: PCs in 18.9..33 {Lc[f'pcs80_{rd}'].between(18.9, 33).sum()}/4; "
          f"participation ratio in 13.59..29.04 {Lc[f'pr_{rd}'].between(13.59, 29.04).sum()}/4")
print(f"location: SNR dimensions in 4.5..13.2 {Lc.snr_dims.between(4.5, 13.2).sum()}/4")

**Result: not reproduced.** The location has 1 or 2 excursion angles above 15 degrees (3.6 to 5.5 expected), needs 3 to 15 PCs for 80% (18.9 to 33) and has participation ratios of 2.3 to 7.4 (13.59 to 29.04); its SNR dimensionality (2 to 9) is in 4.5 to 13.2 for 3 of 4 datasets.

## Variant: the offline movement onset
The paper says only that trials were aligned to movement onset (Methods, Subjects, behavior and data
pre-processing). The file holds two onsets, `move_begins_time` (online, used throughout) and
`offline_move_onset_time`, 25 ms (J) and 52 ms (N) earlier in the median. The claims whose numbers depend most
on the alignment (the jPCA window, the durations of fig3i, the dynamics fitted from -160 ms on) are recomputed
here on both onsets with the same code: the TBF dimensionality (100 folds) and frequencies (fig3g), the TBF
variance (tbf-variance-explained), jPCA R2 (reading b) and variance (jpca-fit-r2, jpca-variance-explained), the
single LDS (single-lds-variance), the single-condition LDS at the projection-chosen k with 20 folds (fig2c,
fig3a), the unwarped and warped correlations of fig3i (reading b, truncate) and the PCs and participation
ratio of the rotation loadings (rotation-orientation-dimensionality, reading a).


In [ ]:
def lds_proj_k(d, g, folds=20):
    """The Methods' projection score for the single-condition LDS dimensionality (as lds_cv, fewer folds)."""
    sc = np.zeros(len(KS))
    for f in range(folds):
        a = ldr.split_halves(d["cond"], g, 0.5)
        W = ldr.cond_average(d["Xt"][a], d["cond"][a])[:, :, ldr.T0:]
        Yb = ldr.cond_average(d["Xt"][~a], d["cond"][~a])[:, :, ldr.T0:]
        den = Yb.var(2, ddof=1).sum(1)
        Up = np.linalg.svd(W, full_matrices=False)[0]
        for i, k in enumerate(KS):
            P = Up[:, :, :k]
            rec = P @ (np.transpose(P, (0, 2, 1)) @ W)
            sc[i] += np.clip(1 - (Yb - rec).var(2, ddof=1).sum(1) / den, 0, None).mean() / folds
    return int(KS[sc.argmax()])

def variant_summary(DSv, tag):
    rows = []
    for n in NAMES:
        d = DSv[n]
        X = d["X"]
        k = ldr.cv_tbf_k(d["Xt"], d["cond"], ldr.rng("fig3g-basis-frequencies"), folds=100)["k"]
        tb = ldr.eig_transform(X, k)
        gr = ldr.groups(tb)
        Br, Lr = ldr.get_basis(X, k)
        r_all = ldr.get_rotations(tb["L"], tb["B"], PLANES[n])
        r_s = ldr.get_rotations(tb["L"], tb["B"], PLANES[n], conds=straight)
        r_c = ldr.get_rotations(tb["L"], tb["B"], PLANES[n], conds=curved)
        X1 = np.concatenate([x[:, ldr.T0:-1] for x in X], 1); X2 = np.concatenate([x[:, ldr.T0 + 1:] for x in X], 1)
        A1 = X2 @ np.linalg.pinv(X1)
        sim = np.full(X.shape, np.nan); sim[:, :, ldr.T0] = X[:, :, ldr.T0]
        for t in range(ldr.T0 + 1, ldr.NT):
            sim[:, :, t] = sim[:, :, t - 1] @ A1.T
        kl = lds_proj_k(d, ldr.rng("fig2c-condition-lds-variance"))
        fits = ldr.lds_batch(X, [kl])
        npair = (kl - 1) // 2
        fr = np.array([np.r_[np.sort(ldr.freq_hz(e[np.imag(e) > 1e-12])), np.full(npair, np.nan)][:npair] for e in fits[kl][0]])
        dt, dc = durations(d, "cond")
        Xw = warped_average(d, dt, ldr.rng("fig3i-duration-warping"))
        Bw, Lw = ldr.get_basis(Xw, k)
        dims = [ldr.pca_dims(np.concatenate([tb["L"][c][:, i].T for c in range(72)])) for f, i in gr if len(i) == 2]
        rows.append(dict(onset=tag, dataset=n, k=k, rot_freqs=[round(f, 2) for f, i in gr if len(i) == 2],
                         tbf_ve=ldr.var_explained(X, ldr.reconstruct(tb["L"], tb["B"])).mean(),
                         jpca_r2_all=r_all["jp"]["R2_Mbest_2D"], jpca_r2_straight=r_s["jp"]["R2_Mbest_2D"],
                         jpca_r2_curved=r_c["jp"]["R2_Mbest_2D"], jpca_ve=r_all["ve"].mean(),
                         single_lds_ve=ldr.var_explained(X, sim).mean(), lds_k=kl,
                         lds_ve=ldr.var_explained(X, fits[kl][1]).mean(),
                         lds_median_freqs=np.round(np.nanmedian(fr, 0), 2).tolist(),
                         fig3i_unwarped=stats.pearsonr(abs(dc - dc.mean()), recoverability(X, Lr, Br))[0],
                         fig3i_warped=stats.pearsonr(abs(dc - dc.mean()), recoverability(Xw, Lw, Bw))[0],
                         rot_pcs80=[p for p, _ in dims], rot_pr=[round(q, 1) for _, q in dims]))
    return pd.DataFrame(rows)

T = time.time()
VAR = [variant_summary(DS, "online")]
del DS
DS_OFF = ldr.build("all", onset="offline_move_onset_time")
for m in MONKEYS:
    print(f"monkey {m}, offline onset: {len(DS_OFF[f'M1-{m}']['cond'])} trials, windows past the trial end dropped "
          f"{DS_OFF[f'M1-{m}']['dropped']}")
VAR.append(variant_summary(DS_OFF, "offline"))
del DS_OFF
VAR = pd.concat(VAR)
pd.set_option("display.width", 250)
print(VAR.round(3).to_string(index=False))
tg = np.array([0.5, 1.5, 2.5, 4.0])
for tag, sub in VAR.groupby("onset", sort=False):
    near = sum(sum(abs(f - tg[np.argmin(abs(tg - f))]) <= 0.1 * tg[np.argmin(abs(tg - f))] for f in fr) for fr in sub.rot_freqs)
    nrot = sum(len(fr) for fr in sub.rot_freqs)
    m1n = sub.set_index("dataset").loc["M1-N"]
    others = sub[sub.dataset != "M1-N"].fig3i_unwarped.between(-0.198, 0.231).sum()
    print(f"{tag}: fig3g rotations within 10% of 0.5/1.5/2.5/4 Hz {near}/{nrot}; tbf VE in 81..100 "
          f"{(sub.tbf_ve >= 0.81).sum()}/4; jPCA R2 all in 0.576..0.803 {sub.jpca_r2_all.between(0.576, 0.803).sum()}/4, "
          f"straight in 0.657..0.968 {sub.jpca_r2_straight.between(0.657, 0.968).sum()}/4, curved in 0.504..0.77 "
          f"{sub.jpca_r2_curved.between(0.504, 0.77).sum()}/4; jPCA VE in 6.3..13.2% {sub.jpca_ve.between(0.063, 0.132).sum()}/4; "
          f"single LDS in 41.4..72.6% {sub.single_lds_ve.between(0.414, 0.726).sum()}/4; LDS k in 5..9 "
          f"{sub.lds_k.between(5, 9).sum()}/4 and VE above 63.9% {(sub.lds_ve >= 0.639).sum()}/4; fig3i M1-N unwarped "
          f"{m1n.fig3i_unwarped:.3f} (in -0.286..-0.234: {-0.286 <= m1n.fig3i_unwarped <= -0.234}), others unwarped in range "
          f"{others}/3, warped in -0.693..-0.468 {sub.fig3i_warped.between(-0.693, -0.468).sum()}/4; rotation PCs to 80% in "
          f"20.7..44 {sum(sum(20.7 <= p <= 44 for p in x) for x in sub.rot_pcs80)}/{nrot}, PR in 15.03..40.48 "
          f"{sum(sum(15.03 <= p <= 40.48 for p in x) for x in sub.rot_pr)}/{nrot}")
print(f"variant in {time.time() - T:.0f} s")


**Result: the outcomes do not depend on the onset.** On the offline onset the TBF chooses the same k (7, 7, 9, 9), the rotation frequencies move by up to 0.3 Hz except the fastest PMd-N pair (3.65 to 4.58 Hz), and 4 of 14 (6 of 14 online) are within 10% of 0.5, 1.5, 2.5 and 4 Hz; the TBF explains 91.9% to 97.4%; jPCA R2 over all conditions is in range in two of four (0.825, 0.581, 0.679, 0.463), straight and curved fits in four of four, and the jPCA variance is 6.4% to 11.4%; the single LDS explains 76.0% to 80.0%, still above 72.6%; the single-condition LDS keeps k = 5, 5, 7, 7 and explains 79.6% to 89.8%; the M1-N unwarped correlation of fig3i is -0.108, and two of four warped correlations are in range; 1 of 14 rotations needs 20.7 to 44 PCs. Each claim recomputed here keeps its outcome.


## Summary

| claim | outcome |
|---|---|
| data-composition | partially reproduced |
| fig3g-basis-frequencies | partially reproduced |
| tbf-variance-explained | reproduced |
| tbf-shuffle-control | reproduced |
| fig3h-recoverability | reproduced |
| fig3i-duration-warping | not reproduced |
| jpca-fit-r2 | partially reproduced |
| jpca-variance-explained | reproduced |
| fig4d-jpca-variance-correlation | not reproduced |
| fig4f-plane-alignment-explains-jpca | partially reproduced |
| single-lds-variance | not reproduced |
| fig2c-condition-lds-variance | reproduced |
| fig3a-eigenvalue-clusters | not reproduced |
| fig3c-eigenvalue-distance-auc | not reproduced |
| suppfig2-eigenvalues-from-kinematics | not reproduced |
| suppfig3-model-comparison | not reproduced |
| fig4b-plane-alignment | partially reproduced |
| suppfig6a-location-alignment | partially reproduced |
| fig5a-frequencies-segregated | partially reproduced |
| suppfig6c-location-distinct-from-rotations | reproduced |
| fig4c-subspace-excursion-angles | partially reproduced |
| rotation-orientation-dimensionality | not reproduced |
| suppfig6b-location-dimensionality | not reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's: The units are the dataset's unsorted threshold crossings (96 per array), not the paper's sorted single and multi-units. Trials are all successful trials of the 72 conditions, aligned on the file's `move_begins_time` (an assumption; the claims that depend most on the onset are recomputed on `offline_move_onset_time` in the variant section, and none changes its outcome). The fig3i durations are run per trial (criterion) and per condition (code); fig4d and fig4f are run with the basis re-extracted from normalised data (Methods) and with the code's input. jPCA is implemented from its description, as its codepack is not in the paper's repository. The LDS dimensionality is scored as the Methods describe (projection) and by the LDS fit, since the code's score does not depend on k. For time: the code's 3/4 split runs 20 folds, the eigenvalue nulls run 150 (halves) and 30 (Poisson) repeats where the Methods use 500, the seed checks of heavy steps use fewer folds or repeats, and the noise for the excursion angles uses 36 half-splits (72 estimates) where the code uses 200.

What was found about the dataset: Conditions and trial counts match the paper (72 conditions; 1595 and 2121 successful trials). The units are unsorted threshold crossings, so the paper's unit counts and its unit selection cannot be checked. The trial table, spikes and hand share a clock (direction decoding 0.80 to 0.92 after onset against 0.20 to 0.23 with shuffled labels; peak hand speed 120 to 140 ms after onset). No two of the ten sessions share trial and spike counts. Two Nitschke sessions (2010-09-20 and 2010-09-22) have no `maze_condition` column.


In [ ]:
print(f"notebook ran in {(time.time() - T_START)/60:.1f} min; peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")